Autores:
* Gabriel Antonio Gomes Moutinho
* Gabriel Henrique Carneiro Amorim

Desafio 2: Ativação e inicialização em redes profundas (sem normalização)
GBC073 — Inteligência Computacional (FACOM/UFU) — Prof. Marcelo Keese Albertini

Entrega oficial:

*   Copie para desafio2/desafio2_nomes.py no seu repositório.
*   Ideia: em vez de decorar Xavier/He, CALIBRAR o ganho numericamente para a ativação escolhida.
*   Se z ~ N(0, 1) e W ~ N(0, s^2), a pré-ativação da próxima camada tem variância
    fan_in * s^2 * E[f(z)^2]
Para mantê-la em 1 camada após camada:  s^2 = 1 / (fan_in * E[f(z)^2]).
* Troque `ativacao` por outra função e a inicialização se ajusta sozinha.
* Rode:  python harness_desafio2.py exemplo_submissao_d2.py --rapido

### Referências da API do PyTorch usadas aqui:
  
  * torch.nn.functional.gelu   https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.gelu.html
  * torch.Tensor.normal_       https://docs.pytorch.org/docs/stable/generated/torch.Tensor.normal_.html
  * torch.Tensor.zero_         https://docs.pytorch.org/docs/stable/generated/torch.Tensor.zero_.html
  * torch.no_grad              https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html
  * torch.Generator            https://docs.pytorch.org/docs/stable/generated/torch.Generator.html
  * torch.nn.init (Xavier, He, calculate_gain — para comparar com a conta feita à mão) https://docs.pytorch.org/docs/stable/nn.init.html
  * outras ativações: relu, leaky_relu, elu, selu, silu em https://docs.pytorch.org/docs/stable/nn.functional.html#non-linear-activation-functions



In [ ]:
%%writefile desafio2_Gabriel_Antonio_Gabriel_Henrique.py

#Autores: Gabriel Antonio Gomes Moutinho, Gabriel Henrique Carneiro Amorim
import math
import torch

def ativacao(x: torch.Tensor) -> torch.Tensor:
    return torch.nn.functional.leaky_relu(x, negative_slope=0.03)
    #função de ativação leaky_relu. Alteramos o slope de 0.01 (padrão) pra 0.03 pois ao fazermos testes obtivemos melhor resultado

_g = torch.Generator().manual_seed(0)
_E_f2 = ativacao(torch.randn(1_000_000, generator=_g)).pow(2).mean().item()

@torch.no_grad()
def inicializar(W: torch.Tensor, b: torch.Tensor,
                fan_in: int, fan_out: int,
                camada: int, n_camadas: int) -> None:

    #primeira camada sem penalização pois não recebe dados de nenhuma camada anterior (recebe x normalizado)
    if camada == 1:
        desvio = math.sqrt(1.0 / fan_in)
    else:
        #camadas 2..n_camadas compensam a energia consumida pela ativação
        #(a última também recebe, e depois ganha o ajuste de logits abaixo)
        desvio = math.sqrt(1.0 / (fan_in * _E_f2))

    # diminuição para ajustar para a última camada, evitando a saturação
    if camada == n_camadas:
        desvio *= 0.5

    W.normal_(0.0, desvio)
    b.zero_()

Writing desafio2_Gabriel_Antonio_Gabriel_Henrique.py


In [48]:
!python harness_desafio2.py desafio2_nomes.py

Desafio 2 — Ativação e inicialização em redes profundas  (torch 2.11.0+cu130, cuda)
MLP largura 256, L ∈ (4, 16, 48), SGD lr=0.05 momento=0.9, 3 época(s), 3 semente(s)


Submissão: desafio2_nomes.py
  mnist_L4           acc=0.967  base=0.963  ref=0.968  s_t=0.78
                     var(pré-ativ) 1ª/meio/última: 1 1.2 0.2 | ‖grad W‖: 1.7 0.67 1.3
  mnist_L16          acc=0.961  base=0.113  ref=0.959  s_t=1.00
                     var(pré-ativ) 1ª/meio/última: 1 1.9 0.3 | ‖grad W‖: 1.6 0.92 1.9
  mnist_L48          acc=0.113  base=0.113  ref=0.101  s_t=0.00
                     var(pré-ativ) 1ª/meio/última: 1 3.6 1 | ‖grad W‖: 1.5 1.9 5.3
  fashion_L4         acc=0.874  base=0.870  ref=0.870  s_t=1.25
                     var(pré-ativ) 1ª/meio/última: 0.98 1.1 0.22 | ‖grad W‖: 2 0.86 1.7
  fashion_L16        acc=0.848  base=0.100  ref=0.852  s_t=0.99
                     var(pré-ativ) 1ª/meio/última: 0.92 1.9 0.26 | ‖grad W‖: 1.8 1.2 2.2
  fashion_L48        acc=0.100  base=0.100  ref=0

In [50]:
!python harness_desafio2.py desafio2_nomes.py --rapido

Desafio 2 — Ativação e inicialização em redes profundas  (torch 2.11.0+cu130, cuda)
MLP largura 256, L ∈ (4, 16, 48), SGD lr=0.05 momento=0.9, 1 época(s), 1 semente(s), modo rápido

  calibrando mnist_L4 ... baseline=0.876 referência=0.890
  calibrando mnist_L16 ... baseline=0.126 referência=0.308
  calibrando mnist_L48 ... baseline=0.126 referência=0.126
  calibrando fashion_L4 ... baseline=0.765 referência=0.808
  calibrando fashion_L16 ... baseline=0.095 referência=0.469
  calibrando fashion_L48 ... baseline=0.095 referência=0.100
  calibrando cifar10_L4 ... baseline=0.340 referência=0.249
  calibrando cifar10_L16 ... baseline=0.100 referência=0.209
  calibrando cifar10_L48 ... baseline=0.100 referência=0.175

Submissão: desafio2_nomes.py
  mnist_L4           acc=0.889  base=0.876  ref=0.890  s_t=0.93
                     var(pré-ativ) 1ª/meio/última: 1 1.2 0.2 | ‖grad W‖: 1.7 0.67 1.3
  mnist_L16          acc=0.586  base=0.126  ref=0.308  s_t=1.25
                     var(pré-ativ)